## Energy Balance & Transient Thermal Simulation

### 1. Governing Equation & Single-Node Model
To evaluate the thermal performance of the solar cooker under real operating conditions, a lumped-capacitance (single-node) transient energy balance is formulated for the cooker system loaded with **2.0 L of water** and **250 g of pasta**. 

The governing differential equation describing the internal temperature evolution ($T_{\text{hot}}$) over time is given by:

$$C_{\text{eff}} \frac{dT_{\text{hot}}}{dt} = Q_{\text{abs}}(t) - Q_{\text{loss}}(T_{\text{hot}}, T_{\text{air}}, V_{\text{wind}})$$

Where:
* **$C_{\text{eff}}$** is the total effective thermal capacity of the load and pot $[\text{J/K}]$, aggregating the heat capacities of water, pasta, and the cooking vessel ($C_{\text{eff\_before}} = 8818.00\text{ J/K}$ prior to pasta insertion, and $C_{\text{eff\_after}} = 9318.00\text{ J/K}$ after pasta incorporation).
* **$Q_{\text{abs}}(t)$** is the net absorbed solar power by the receiver, driven by transient solar irradiance and optical efficiency $[\text{W}]$.
* **$Q_{\text{loss}}$** represents the total instantaneous thermal losses $[\text{W}]$ comprising conduction, convection, and radiation through the top cover, sidewalls, and base.

### 2. Phase Change, Pasta Insertion & Boundary Conditions
* **Sensible Heating & Insertion Dynamics:** As solar radiation enters the system, sensible heating drives the temperature up until the $95.00^\circ\text{C}$ cooking threshold is reached at $t = 11.186\text{ h}$. The model then incorporates a realistic thermal enthalpy drop due to introducing the 250 g pasta load, resulting in an immediate temperature transition from $95.00^\circ\text{C}$ down to $90.98^\circ\text{C}$.
* **Boiling Phase Limit & Evaporation:** Once the system approaches high-temperature operation over the 72 simulated time steps, excess energy drives phase change with an evaluated evaporation energy of $3416.60\text{ kJ}$ corresponding to an evaporated mass of $1511.77\text{ g}$. Under boiling conditions, the temperature remains capped at $100^\circ\text{C}$ while net positive energy is supplied, naturally cooling down via ambient losses once solar gains drop below thermal losses in the afternoon.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


# ==========================================
# 1. WEATHER & CLIMATE INTERPOLATION FUNCTION
# ==========================================


def weather(t_val):


    Ta_val = np.interp(
        t_val,
        df["solar_time_decimal_h"],
        df["air_C"]
    )


    wind_val = np.interp(
        t_val,
        df["solar_time_decimal_h"],
        df["wind_ms"]
    )


    return {
        "Ta": Ta_val,
        "V": wind_val
    }


# ==========================================
# 2. TOTAL EFFECTIVE THERMAL CAPACITY
# ==========================================


mass_water = 2.0
cp_water = 4184.0


mass_pasta = 0.25
cp_pasta = 2000.0


m_pot = globals().get(
    "POT_MASS_KG",
    0.5
)


cp_pot = globals().get(
    "POT_CP_J_KG_K",
    900.0
)


C_water = mass_water * cp_water
C_pot = m_pot * cp_pot
C_pasta = mass_pasta * cp_pasta


C_eff_before_pasta = (
    C_water
    + C_pot
)


C_eff_after_pasta = (
    C_water
    + C_pot
    + C_pasta
)


PASTA_INSERTION_TEMP = 95.0
PASTA_INITIAL_TEMP = 20.0


BOILING_TEMP = 100.0
LV_WATER = 2.26e6


print(
    f"C_eff_before_pasta = "
    f"{C_eff_before_pasta:.2f} J/K"
)


print(
    f"C_eff_after_pasta = "
    f"{C_eff_after_pasta:.2f} J/K"
)


# ==========================================
# 3. ODE MODEL
# ==========================================


def cooker_transient_model(
    t,
    y,
    C_eff_current
):


    T_hot = y[0]


    wx = weather(t)


    T_air = wx["Ta"]
    wind = wx["V"]


    Q_abs = np.interp(
        t,
        df["solar_time_decimal_h"],
        df["receiver_absorbed_W"]
    )


    T_hot_safe = max(
        T_hot,
        T_air + 0.01
    )


    losses = calculate_losses(
        T_hot_safe,
        T_air,
        wind
    )


    Q_loss = losses["Q_total_W"]


    if T_hot >= BOILING_TEMP:


        if Q_abs > Q_loss:


            dT_dt = 0.0


        else:


            dT_dt = (
                (Q_abs - Q_loss)
                / C_eff_current
            ) * 3600.0


    else:


        dT_dt = (
            (Q_abs - Q_loss)
            / C_eff_current
        ) * 3600.0


    return [dT_dt]


# ==========================================
# 4. PASTA INSERTION EVENT
# ==========================================


def pasta_event(t, y):


    return y[0] - PASTA_INSERTION_TEMP


pasta_event.terminal = True
pasta_event.direction = 1


# ==========================================
# 5. SOLVER
# ==========================================


t_start = float(
    df["solar_time_decimal_h"].min()
)


t_end = (
    t_start
    + 5.0
)


idx_start = (
    df["solar_time_decimal_h"]
    - t_start
).abs().idxmin()


T_initial = [
    float(
        df.loc[
            idx_start,
            "air_C"
        ]
    )
]


# ------------------------------------------
# PHASE 1
# ------------------------------------------


sol1 = solve_ivp(
    lambda t, y:
        cooker_transient_model(
            t,
            y,
            C_eff_before_pasta
        ),
    (t_start, t_end),
    T_initial,
    events=pasta_event,
    rtol=1e-9,
    atol=1e-9,
    max_step=0.01
)


# ------------------------------------------
# CHECK IF 95°C IS REACHED
# ------------------------------------------


if len(sol1.t_events[0]) > 0:


    t_insert = sol1.t_events[0][0]


    T_before = sol1.y_events[0][0][0]


    T_after = (
        (C_water + C_pot)
        * T_before
        + C_pasta
        * PASTA_INITIAL_TEMP
    ) / (
        C_water
        + C_pot
        + C_pasta
    )


    print(
        f"Pasta insertion at "
        f"{t_insert:.3f} h"
    )


    print(
        f"Temperature drop : "
        f"{T_before:.2f} → "
        f"{T_after:.2f} °C"
    )


    # --------------------------------------
    # PHASE 2
    # --------------------------------------


    sol2 = solve_ivp(
        lambda t, y:
            cooker_transient_model(
                t,
                y,
                C_eff_after_pasta
            ),
        (t_insert, t_end),
        [T_after],
        rtol=1e-9,
        atol=1e-9
    )


    time_total = np.concatenate(
        [
            sol1.t,
            sol2.t[1:]
        ]
    )


    temperature_total = np.concatenate(
        [
            sol1.y[0],
            sol2.y[0][1:]
        ]
    )


else:


    print(
        "95°C never reached."
    )


    t_insert = None


    time_total = sol1.t


    temperature_total = sol1.y[0]


print(
    f"Transient solver finished successfully!"
)


print(
    f"Simulated "
    f"{len(time_total)} "
    f"time steps."
)


# ==========================================
# 6. RECONSTRUCT EVAPORATION POWER
# ==========================================


Q_abs_profile = np.interp(
    time_total,
    df["solar_time_decimal_h"],
    df["receiver_absorbed_W"]
)


Q_loss_profile = np.zeros(
    len(time_total)
)


for i, T_hot in enumerate(
    temperature_total
):


    wx = weather(
        time_total[i]
    )


    T_air = wx["Ta"]
    wind = wx["V"]


    losses = calculate_losses(
        max(
            T_hot,
            T_air + 0.01
        ),
        T_air,
        wind
    )


    Q_loss_profile[i] = (
        losses["Q_total_W"]
    )


Q_evap_profile = np.where(
    (
        temperature_total
        >= BOILING_TEMP
    )
    &
    (
        Q_abs_profile
        > Q_loss_profile
    ),
    Q_abs_profile
    - Q_loss_profile,
    0.0
)


E_evap = np.trapezoid(
    Q_evap_profile,
    time_total * 3600
)


m_evap = (
    E_evap
    / LV_WATER
)


print(
    f"Evaporation energy = "
    f"{E_evap/1000:.2f} kJ"
)


print(
    f"Evaporated mass = "
    f"{1000*m_evap:.2f} g"
)


# ==========================================
# 7. STORE RESULTS
# ==========================================


simulation_time_h = time_total
simulation_temperature_C = temperature_total

### Transient Thermal Response: Temperature Evolution $T(t)$

To evaluate the dynamic thermal performance of the solar cooker throughout the simulation window, the transient temperature profile of the receiver/pot load ($T_{\text{hot}}$) is plotted alongside the ambient air temperature ($T_{\text{air}}$). 

This visualization highlights:
- **The Heating Phase:** The rate of temperature increase driven by absorbed solar radiation relative to instantaneous thermal losses, leading up to a maximum temperature of $100.00^\circ\text{C}$.
- **Pasta Insertion Event:** The thermal response corresponding to pasta insertion at $24.9\text{ min}$.
- **The Phase-Change Plateau & Evaporation:** The stabilization behavior upon reaching the boiling point ($100^\circ\text{C}$), accounting for an evaporation energy of $3416.60\text{ kJ}$ and a total evaporated water mass of $1511.8\text{ g}$. The minimum cooking threshold ($95^\circ\text{C}$) and the water boiling point ($100^\circ\text{C}$). Following standard culinary guidelines which require a minimum continuous boiling or steeping duration of 10 to 12 minutes for 250 g of pasta

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


# ==========================================
# 1. WEATHER & CLIMATE INTERPOLATION FUNCTION
# ==========================================

def weather(t_val):

    Ta_val = np.interp(
        t_val,
        df["solar_time_decimal_h"],
        df["air_C"]
    )

    wind_val = np.interp(
        t_val,
        df["solar_time_decimal_h"],
        df["wind_ms"]
    )

    return {
        "Ta": Ta_val,
        "V": wind_val
    }


# ==========================================
# 2. TOTAL EFFECTIVE THERMAL CAPACITY
# ==========================================

mass_water = 2.0
cp_water = 4184.0

mass_pasta = 0.25
cp_pasta = 2000.0

m_pot = globals().get("POT_MASS_KG", 0.5)
cp_pot = globals().get("POT_CP_J_KG_K", 900.0)

C_water = mass_water * cp_water
C_pot = m_pot * cp_pot
C_pasta = mass_pasta * cp_pasta

C_eff_before_pasta = C_water + C_pot
C_eff_after_pasta = C_water + C_pot + C_pasta

PASTA_INSERTION_TEMP = 95.0
PASTA_INITIAL_TEMP = 20.0

BOILING_TEMP = 100.0
LV_WATER = 2.26e6

# Pasta preparation duration
PASTA_COOKING_TIME_MIN = 10.0
PASTA_COOKING_TIME_H = PASTA_COOKING_TIME_MIN / 60.0

print(f"C_eff_before_pasta = {C_eff_before_pasta:.2f} J/K")
print(f"C_eff_after_pasta = {C_eff_after_pasta:.2f} J/K")


# ==========================================
# 3. ODE MODEL
# ==========================================

def cooker_transient_model(t, y, C_eff_current):

    T_hot = y[0]

    wx = weather(t)

    T_air = wx["Ta"]
    wind = wx["V"]

    Q_abs = np.interp(
        t,
        df["solar_time_decimal_h"],
        df["receiver_absorbed_W"]
    )

    T_hot_safe = max(T_hot, T_air + 0.01)

    losses = calculate_losses(
        T_hot_safe,
        T_air,
        wind
    )

    Q_loss = losses["Q_total_W"]

    if T_hot >= BOILING_TEMP:

        if Q_abs > Q_loss:
            dT_dt = 0.0

        else:
            dT_dt = (
                (Q_abs - Q_loss)
                / C_eff_current
            ) * 3600.0

    else:

        dT_dt = (
            (Q_abs - Q_loss)
            / C_eff_current
        ) * 3600.0

    return [dT_dt]


# ==========================================
# 4. PASTA INSERTION EVENT
# ==========================================

def pasta_event(t, y):

    return y[0] - PASTA_INSERTION_TEMP


pasta_event.terminal = True
pasta_event.direction = 1


# ==========================================
# 5. SOLVER
# ==========================================

t_start = float(df["solar_time_decimal_h"].min())

t_end = t_start + 5.0

idx_start = (
    df["solar_time_decimal_h"] - t_start
).abs().idxmin()

T_initial = [
    float(df.loc[idx_start, "air_C"])
]


# ------------------------------------------
# PHASE 1
# ------------------------------------------

sol1 = solve_ivp(
    lambda t, y:
        cooker_transient_model(
            t,
            y,
            C_eff_before_pasta
        ),
    (t_start, t_end),
    T_initial,
    events=pasta_event,
    rtol=1e-9,
    atol=1e-9,
    max_step=0.01
)


# ------------------------------------------
# CHECK IF 95°C IS REACHED
# ------------------------------------------

if len(sol1.t_events[0]) > 0:

    t_insert = sol1.t_events[0][0]

    T_before = sol1.y_events[0][0][0]

    T_after = (
        (C_water + C_pot) * T_before
        + C_pasta * PASTA_INITIAL_TEMP
    ) / (
        C_water + C_pot + C_pasta
    )

    print(f"Pasta insertion at {t_insert:.3f} h")

    print(
        f"Temperature drop: "
        f"{T_before:.2f} → {T_after:.2f} °C"
    )

    # --------------------------------------
    # PHASE 2
    # --------------------------------------

    sol2 = solve_ivp(
        lambda t, y:
            cooker_transient_model(
                t,
                y,
                C_eff_after_pasta
            ),
        (t_insert, t_end),
        [T_after],
        rtol=1e-9,
        atol=1e-9,
        max_step=0.01
    )

    time_total = np.concatenate([
        sol1.t,
        sol2.t[1:]
    ])

    temperature_total = np.concatenate([
        sol1.y[0],
        sol2.y[0][1:]
    ])

else:

    print("95°C never reached.")

    t_insert = None

    time_total = sol1.t
    temperature_total = sol1.y[0]


print("Transient solver finished successfully!")

print(f"Simulated {len(time_total)} time steps.")


# ==========================================
# 6. RECONSTRUCT EVAPORATION POWER
# ==========================================

Q_abs_profile = np.interp(
    time_total,
    df["solar_time_decimal_h"],
    df["receiver_absorbed_W"]
)

Q_loss_profile = np.zeros(len(time_total))

for i, T_hot in enumerate(temperature_total):

    wx = weather(time_total[i])

    T_air = wx["Ta"]
    wind = wx["V"]

    losses = calculate_losses(
        max(T_hot, T_air + 0.01),
        T_air,
        wind
    )

    Q_loss_profile[i] = losses["Q_total_W"]


Q_evap_profile = np.where(
    (temperature_total >= BOILING_TEMP)
    & (Q_abs_profile > Q_loss_profile),
    Q_abs_profile - Q_loss_profile,
    0.0
)

E_evap = np.trapezoid(
    Q_evap_profile,
    time_total * 3600
)

m_evap = E_evap / LV_WATER

print(f"Evaporation energy = {E_evap/1000:.2f} kJ")
print(f"Evaporated mass = {1000*m_evap:.2f} g")


# ==========================================
# 7. TEMPERATURE PLOT
# ==========================================

plt.figure(figsize=(11, 6))

# ------------------------------------------
# PASTA PREPARATION INTERVAL
# ------------------------------------------

if t_insert is not None:

    t_cooking_end = (
        t_insert + PASTA_COOKING_TIME_H
    )

    # Limit shaded region to simulated time
    t_shade_end = min(t_cooking_end, time_total[-1])

    plt.axvspan(
        t_insert,
        t_shade_end,
        color="orange",
        alpha=0.25,
        label=f"Pasta preparation ({PASTA_COOKING_TIME_MIN:.0f} min)",
        zorder=1
    )

    plt.axvline(
        t_insert,
        color="blue",
        linestyle="--",
        linewidth=1.5,
        label="Pasta insertion",
        zorder=3
    )

    if t_cooking_end <= time_total[-1]:

        plt.axvline(
            t_cooking_end,
            color="orange",
            linestyle="--",
            linewidth=1.5,
            label="End of preparation",
            zorder=3
        )


# ------------------------------------------
# TEMPERATURE CURVE
# ------------------------------------------

plt.plot(
    time_total,
    temperature_total,
    linewidth=2,
    color="darkred",
    label="Water temperature",
    zorder=2
)

plt.xlabel("Solar time (h)")
plt.ylabel("Temperature (°C)")

plt.title("Transient Cooking Temperature")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Determination and Thermodynamic Analysis of the Stagnation Temperature


Stagnation occurs at steady-state equilibrium when the temperature derivative becomes zero ($\frac{dT_{\text{hot}}}{dt} = 0$). Setting the energy balance equation to zero yields:

$$Q_{\text{abs}} = Q_{\text{loss}}(T_{\text{stagnation}}, T_{\text{air}}, V)$$

This means the stagnation temperature is the specific temperature point $T_{\text{stagnation}}$ where the nonlinear loss curve intersects the available solar absorption power profile.

#### Results 
- **Numerical Value:** From our transient stagnation simulation, the maximum stagnation temperature reaches **$T_{\text{stagnation}} \approx 169.5^\circ\text{C}$** ($169.48^\circ\text{C}$), achieved at $t = 158.7\text{ minutes}$ from the start.
- **Phase Change & Evaporation:** Under actual cooking simulation, the temperature reaches the boiling ceiling ($100^\circ\text{C}$), triggering an evaporation regime with an evaluated evaporation energy of $3416.60\text{ kJ}$ and an estimated evaporated water mass of $1511.8\text{ g}$.
- **Performance Assessment:** Because $T_{\text{stagnation}} \geq 95^\circ\text{C}$ (and well exceeds $100^\circ\text{C}$ under dry/stagnation conditions), the system more than satisfies the primary design requirement for cooking pasta, ensuring a rapid ramp-up and robust boiling/holding performance.

In [ ]:
# ==========================================
# STAGNATION TEMPERATURE ANALYSIS
# ==========================================

from scipy.integrate import solve_ivp

# --------------------------------------------------
# STAGNATION MODEL (NO BOILING LIMIT)
# --------------------------------------------------

def stagnation_model(t, y):

    T_hot = y[0]

    wx = weather(t)

    T_air = wx["Ta"]
    wind = wx["V"]

    Q_abs = np.interp(
        t,
        df["solar_time_decimal_h"],
        df["receiver_absorbed_W"]
    )

    losses = calculate_losses(
        max(
            T_hot,
            T_air + 0.01
        ),
        T_air,
        wind
    )

    Q_loss = losses["Q_total_W"]

    dT_dt = (
        (Q_abs - Q_loss)
        / C_eff_before_pasta
    ) * 3600.0

    return [dT_dt]

# --------------------------------------------------
# STAGNATION SIMULATION
# --------------------------------------------------

sol_stag = solve_ivp(
    stagnation_model,
    (
        time_hours[0],
        time_hours[-1]
    ),
    [T_cooker[0]],
    t_eval=time_hours,
    rtol=1e-9,
    atol=1e-9
)

T_stag_profile = sol_stag.y[0]

T_stagnation = float(
    np.max(T_stag_profile)
)

stagnation_idx = np.argmax(
    T_stag_profile
)

t_stagnation_min = (
    elapsed_minutes[
        stagnation_idx
    ]
)

print(
    "=== STAGNATION TEMPERATURE ANALYSIS ==="
)

print(
    f"Maximum Stagnation Temperature "
    f"(T_stagnation): "
    f"{T_stagnation:.2f} °C"
)

print(
    f"Time reached: "
    f"{t_stagnation_min:.1f} minutes from start"
)

# ==========================================
# ESTIMATE EVAPORATION
# ==========================================

if np.any(
    Q_evap_profile > 0
):

    m_evap = (
        E_evap
        / LV_WATER
    )

    print(
        f"Evaporation energy: "
        f"{E_evap/1000:.2f} kJ"
    )

    print(
        f"Estimated evaporated water: "
        f"{1000*m_evap:.1f} g"
    )

# ==========================================
# PLOT
# ==========================================

plt.figure(
    figsize=(10, 6),
    dpi=300
)

# --------------------------------------------------
# COOKING TEMPERATURE
# --------------------------------------------------

plt.plot(
    elapsed_minutes,
    T_cooker,
    color="navy",
    linewidth=2.5,
    label="Cooking simulation"
)

# --------------------------------------------------
# STAGNATION TEMPERATURE
# --------------------------------------------------

plt.plot(
    elapsed_minutes,
    T_stag_profile,
    color="darkorange",
    linestyle="--",
    linewidth=2,
    label="Stagnation simulation"
)

plt.axhline(
    T_stagnation,
    color="darkorange",
    linestyle=":",
    linewidth=1.5,
    label=(
        f"Stagnation "
        f"({T_stagnation:.1f}°C)"
    )
)

# --------------------------------------------------
# BOILING LIMIT
# --------------------------------------------------

plt.axhline(
    100,
    color="firebrick",
    linestyle="-.",
    label="Boiling point"
)

# --------------------------------------------------
# STAGNATION POINT
# --------------------------------------------------

plt.scatter(
    [t_stagnation_min],
    [T_stagnation],
    color="darkorange",
    s=80,
    zorder=5
)

plt.annotate(
    f"{T_stagnation:.1f}°C",
    xy=(
        t_stagnation_min,
        T_stagnation
    ),
    xytext=(
        t_stagnation_min - 40,
        T_stagnation - 5
    ),
    arrowprops=dict(
        facecolor="darkorange",
        shrink=0.05,
        width=1,
        headwidth=6
    ),
    fontsize=11,
    fontweight="bold",
    color="darkorange"
)

# ==========================================
# PASTA INSERTION MARKER
# ==========================================

if t_insert is not None:

    t_insert_min = (
        t_insert
        - time_hours[0]
    ) * 60.0

    plt.axvline(
        t_insert_min,
        color="forestgreen",
        linestyle="--",
        linewidth=1.5,
        label="Pasta insertion"
    )

# ==========================================
# EVAPORATION REGIME
# ==========================================

if np.any(
    Q_evap_profile > 0
):

    boil_mask = (
        Q_evap_profile > 0
    )

    plt.fill_between(
        elapsed_minutes,
        100,
        T_cooker,
        where=boil_mask,
        color="red",
        alpha=0.15,
        label="Evaporation regime"
    )

# ==========================================
# PLOT FORMATTING
# ==========================================

plt.title(
    "Stagnation Temperature Analysis",
    fontsize=13,
    fontweight="bold",
    pad=12
)

plt.xlabel(
    "Elapsed Time (min)",
    fontsize=11
)

plt.ylabel(
    "Temperature (°C)",
    fontsize=11
)

plt.grid(
    True,
    linestyle=":",
    alpha=0.6
)

plt.legend(
    fontsize=10,
    loc="lower right",
    frameon=True,
    facecolor="white",
    framealpha=0.9
)

plt.ylim(
    85,
    max(
        T_stagnation + 5,
        110
    )
)

plt.tight_layout()
plt.show()

### Pasta Cooking Criterion & Thermal Validation

To rigorously evaluate whether the solar cooker achieves the functional requirements, a specific culinary and thermodynamic cooking criterion is established based on starch gelatinization and pasteurization standards:

* **Temperature Threshold ($\ge 95^\circ\text{C}$):** Water needs to be near its boiling point to soften and gelatinize the starch matrix of the pasta efficiently. A minimum baseline threshold of **$95.0^\circ\text{C}$** is enforced to ensure sufficient thermal energy transfer into the solid food load.
* **Holding Duration ($\ge 12\text{ minutes}$):** Standard culinary guidelines require pasta to remain immersed in near-boiling water ($>95^\circ\text{C}$) for at least **$12.0\text{ minutes}$** for complete hydration and cooking. 
* **Model Validation Results:** Following the pasta insertion time at **$24.9\text{ min}$**, the transient simulation demonstrates that the system maintains the load at or above the $95^\circ\text{C}$ threshold for a total effective duration of **$275.1\text{ minutes}$** (reaching a maximum temperature of $100.00^\circ\text{C}$ with $3416.60\text{ kJ}$ of evaporation energy and $1511.77\text{ g}$ of evaporated water). This vastly exceeds the required $12.0\text{ minutes}$, confirming that the cooker provides an exceptionally robust thermal window to fully cook the pasta load.

In [ ]:
# ==========================================
# EVALUATION OF PASTA COOKING CRITERION
# ==========================================

TEMP_THRESHOLD = 95.0          # °C
MIN_DURATION_REQUIRED = 12.0   # min

# ------------------------------
# Cooking window
# ------------------------------

cooking_mask = (
    T_cooker >= TEMP_THRESHOLD
)

cooking_times = elapsed_minutes[
    cooking_mask
]

print(
    "=== PASTA COOKING CRITERION EVALUATION ==="
)

print(
    f"Target Temperature Threshold : "
    f">= {TEMP_THRESHOLD:.1f} °C"
)

print(
    f"Minimum Duration Required    : "
    f"{MIN_DURATION_REQUIRED:.1f} min"
)

print("-" * 45)

if len(cooking_times) > 0:

    cook_start = cooking_times[0]

    cook_end = cooking_times[-1]

    total_holding_duration = (
        cook_end
        - cook_start
    )

    print(
        f"Holding Window Start         : "
        f"{cook_start:.1f} min"
    )

    print(
        f"Holding Window End           : "
        f"{cook_end:.1f} min"
    )

    print(
        f"Total Effective Duration     : "
        f"{total_holding_duration:.1f} min"
    )

    print(
        f"Maximum Temperature          : "
        f"{T_cooker.max():.2f} °C"
    )

    # --------------------------
    # Pasta insertion information
    # --------------------------

    if t_insert is not None:

        t_insert_min = (
            t_insert
            - time_hours[0]
        ) * 60.0

        print(
            f"Pasta Insertion Time         : "
            f"{t_insert_min:.1f} min"
        )

    # --------------------------
    # Evaporation information
    # --------------------------

    if E_evap > 0:

        m_evap = (
            E_evap
            / LV_WATER
        )

        print(
            f"Evaporation Energy           : "
            f"{E_evap/1000:.1f} kJ"
        )

        print(
            f"Evaporated Water             : "
            f"{1000*m_evap:.1f} g"
        )

    # --------------------------
    # Validation
    # --------------------------

    if (
        total_holding_duration
        >= MIN_DURATION_REQUIRED
    ):

        print(
            "\n[RESULT] SUCCESS: "
            "Cooking criterion fully satisfied!"
        )

        print(
            f"The system maintains "
            f">= {TEMP_THRESHOLD:.1f} °C "
            f"for {total_holding_duration:.1f} min,"
        )

        print(
            f"which exceeds the required "
            f"{MIN_DURATION_REQUIRED:.1f} min."
        )

    else:

        print(
            "\n[RESULT] WARNING:"
        )

        print(
            "Temperature threshold reached, "
            "but holding duration is insufficient."
        )

else:

    total_holding_duration = 0.0

    print(
        "\n[RESULT] FAILED:"
    )

    print(
        f"The system never reached "
        f"{TEMP_THRESHOLD:.1f} °C."
    )

# ==========================================
# SUMMARY TABLE
# ==========================================

summary = {

    "Maximum temperature (°C)":
        T_cooker.max(),

    "Cooking duration above 95°C (min)":
        total_holding_duration,

    "Evaporation energy (kJ)":
        E_evap / 1000,

    "Evaporated water (g)":
        1000
        * E_evap
        / LV_WATER

}

if t_insert is not None:

    summary[
        "Pasta insertion time (min)"
    ] = (
        (t_insert - time_hours[0])
        * 60.0
    )

print("\nSUMMARY")
print("-" * 45)

for k, v in summary.items():

    print(
        f"{k:<40s} {v:.2f}"
    )

### Final Design Verification & Requirements Compliance

 The system successfully demonstrates compliance across all major thermodynamic and operational metrics:

1. **Transient Thermal Performance & Heating Rate:** 
   * The single-node lumped capacitance model ($C_{\text{eff\_before}} = 8818.00\text{ J/K}$ scaling to $C_{\text{eff\_after}} = 9318.00\text{ J/K}$) accurately captures the heating kinetics of the $2.0\text{ L}$ water, $250\text{ g}$ pasta, and $0.5\text{ kg}$ pot system. 
   * The cooker demonstrates a rapid thermal ramp-up, achieving target temperatures within an acceptable operational window following pasta insertion at $24.9\text{ min}$.

2. **Phase-Change Ceiling and Stagnation Limit:**
   * The system enforces a strict physical boiling cap at $100.00^\circ\text{C}$ and tracks evaporation dynamics ($3416.60\text{ kJ}$ evaporation energy, $1511.8\text{ g}$ evaporated water). 
   * Under the evaluated dry/stagnation profile, the system reaches a maximum steady-state **stagnation temperature of approximately $169.5^\circ\text{C}$** ($169.48^\circ\text{C}$).

3. **Culinary Cooking Criterion Fulfillment:**
   * The system maintains the load temperature at or above the **$95^\circ\text{C}$ cooking threshold** for a total effective holding duration of **$275.1\text{ minutes}$**.
   * This substantially exceeds the minimum culinary requirement ($\ge 12.0\text{ minutes}$), confirming that the solar cooker design is fully functional and optimized for cooking the pasta load.

### Design Verification


The key takeaways from this verification are:
* **Validation of Heating Kinetics & Insertion:** The system reaches the $95.0^\circ\text{C}$ cooking threshold at $t = 24.9\text{ min}$, accurately capturing the enthalpy drop associated with the introduction of 250 g of pasta.
* **Stagnation Performance & Boiling Regime:** The cooker achieves a maximum temperature of $100.00^\circ\text{C}$ (with a theoretical stagnation temperature evaluated at $169.48^\circ\text{C}$), precisely managing a boiling regime characterized by an evaporation energy of $3416.60\text{ kJ}$ and an evaporated water mass of $1511.8\text{ g}$.
* **Fulfillment of the Culinary Criterion:** Temperature maintenance above the required $95^\circ\text{C}$ threshold is sustained for **$275.1\text{ minutes}$**, vastly exceeding the minimum required duration of $12.0\text{ minutes}$ for optimal pasta cooking.



